# Milestone 4 Instructions — Luke

## Your Responsibilities
1. **Task 5: Unit Test + Refactor** — Refactor health-status classification into a testable pure function and write pytest unit tests (Branch: `feat/unit-tests`)
2. **Task 7: CHANGELOG (partial)** — Write the Added / Changed / Fixed / Known Issues sections for `## [v0.4.0]` (Branch: `docs/changelog-v0.4.0`)

**Key files you create/modify:** `src/components/health_status.py`, `src/pages/company.py`, `tests/test_health_status.py`, `CHANGELOG.md`

> **IMPORTANT:** Task 5 (unit tests + refactor) should be completed and merged before Task 7 (CHANGELOG) since the CHANGELOG references the refactored functions.

---

## Part 1: Unit Test + Refactor (Task 5)

### Background

In the current `src/pages/company.py`, there are **4 nearly identical status functions** (`p2_npm_status`, `p2_roe_status`, `p2_current_ratio_status`, `p2_debt_equity_status`). Each one:
1. Gets filtered data
2. Extracts a numeric value
3. Compares against hardcoded thresholds
4. Returns a healthy/warning/danger UI element

Additionally, `p2_cash_flows` contains an inline `fmt()` closure for currency formatting.

**The refactoring plan:**
- Extract `classify_health(value, healthy, warning, higher_is_better)` — a pure function that returns `"healthy"`, `"warning"`, or `"danger"`
- Extract `format_currency(value)` — a pure function that returns a formatted string like `"$1,234M"`
- Both go into a new module: `src/components/health_status.py`
- Update `company.py` to import and use these functions
- Write comprehensive pytest tests in `tests/test_health_status.py`

---

### Step 1: Create the feature branch

```bash
git checkout main
git pull origin main
git checkout -b feat/unit-tests
git push origin feat/unit-tests
```

---

### Step 2: Create `src/components/health_status.py`

Create a new file `src/components/health_status.py` with two pure functions extracted from `company.py`. These functions have **no Shiny dependencies** — they take simple inputs and return simple outputs, making them easy to test.

In [ ]:
"""Pure helper functions for financial health classification and formatting."""


def classify_health(value: float, healthy: float, warning: float, higher_is_better: bool = True) -> str:
    """Classify a financial metric value into a health status.

    Parameters
    ----------
    value : float
        The metric value to classify.
    healthy : float
        Threshold for "healthy" status.
    warning : float
        Threshold for "warning" status (between healthy and danger).
    higher_is_better : bool
        If True, values >= healthy are healthy. If False, values <= healthy are healthy.

    Returns
    -------
    str
        One of "healthy", "warning", or "danger".
    """
    if higher_is_better:
        if value >= healthy:
            return "healthy"
        elif value >= warning:
            return "warning"
        else:
            return "danger"
    else:
        if value <= healthy:
            return "healthy"
        elif value <= warning:
            return "warning"
        else:
            return "danger"


def format_currency(value: float) -> str:
    """Format a dollar value in millions with sign.

    Parameters
    ----------
    value : float
        Dollar amount in millions.

    Returns
    -------
    str
        Formatted string like "$1,234M" or "-$567M".
    """
    if value < 0:
        return f"-${abs(value):,.0f}M"
    return f"${value:,.0f}M"

**What this code does:**
- `classify_health()` replaces the repeated if/elif/else threshold logic in all 4 status functions. It supports both "higher is better" metrics (NPM, ROE, Current Ratio) and "lower is better" metrics (Debt/Equity).
- `format_currency()` replaces the inline `fmt()` closure inside `p2_cash_flows`. It formats a number with commas, `$` sign, and `M` suffix, handling negatives with a leading `-$`.

> **Note:** Make sure `src/components/__init__.py` exists (it should already exist from M3). If not, create an empty `__init__.py` in `src/components/`.

---

### Step 3: Update `src/pages/company.py`

Now refactor `company.py` to use the new pure functions. There are **6 changes** to make:

1. Add the import at the top
2. Add a shared `_STATUS_ICONS` dict and `_status_badge()` helper inside `company_server()`
3. Refactor `p2_npm_status` to use `classify_health`
4. Refactor `p2_roe_status` to use `classify_health`
5. Refactor `p2_current_ratio_status` to use `classify_health`
6. Refactor `p2_debt_equity_status` to use `classify_health` (with `higher_is_better=False`)
7. Refactor `p2_cash_flows` to use `format_currency` instead of inline `fmt()`

---

#### Change 1: Add the import

At the top of `src/pages/company.py`, add this import alongside the existing component imports:

**BEFORE:**
```python
from components.empty_chart import empty_chart
```

**AFTER:**
```python
from components.empty_chart import empty_chart
from components.health_status import classify_health, format_currency
```

---

#### Change 2: Add `_STATUS_ICONS` and `_status_badge()` inside `company_server()`

Inside the `company_server()` function, add these right before `p2_npm_status`:

```python
    # Status icon mapping for health classification
    _STATUS_ICONS = {"healthy": "\u2713", "warning": "!", "danger": "\u2717"}

    def _status_badge(status: str):
        icon = _STATUS_ICONS[status]
        return ui.tags.span(icon, class_=f"kpi-status {status}")
```

This shared helper converts a status string into the UI badge element, eliminating the repeated `ui.tags.span(...)` calls.

---

#### Change 3: Refactor `p2_npm_status`

**BEFORE:**
```python
    @render.ui
    def p2_npm_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Net Profit Margin"].iloc[0]
        if value >= 10:
            return ui.tags.span("\u2713", class_="kpi-status healthy")
        elif value >= 0:
            return ui.tags.span("!", class_="kpi-status warning")
        else:
            return ui.tags.span("\u2717", class_="kpi-status danger")
```

**AFTER:**
```python
    @render.ui
    def p2_npm_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Net Profit Margin"].iloc[0]
        return _status_badge(classify_health(value, healthy=10.0, warning=0.0))
```

---

#### Change 4: Refactor `p2_roe_status`

**BEFORE:**
```python
    @render.ui
    def p2_roe_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["ROE"].iloc[0]
        if value >= 15:
            return ui.tags.span("\u2713", class_="kpi-status healthy")
        elif value >= 0:
            return ui.tags.span("!", class_="kpi-status warning")
        else:
            return ui.tags.span("\u2717", class_="kpi-status danger")
```

**AFTER:**
```python
    @render.ui
    def p2_roe_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["ROE"].iloc[0]
        return _status_badge(classify_health(value, healthy=15.0, warning=0.0))
```

---

#### Change 5: Refactor `p2_current_ratio_status`

**BEFORE:**
```python
    @render.ui
    def p2_current_ratio_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Current Ratio"].iloc[0]
        if value >= 1.5:
            return ui.tags.span("\u2713", class_="kpi-status healthy")
        elif value >= 1.0:
            return ui.tags.span("!", class_="kpi-status warning")
        else:
            return ui.tags.span("\u2717", class_="kpi-status danger")
```

**AFTER:**
```python
    @render.ui
    def p2_current_ratio_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Current Ratio"].iloc[0]
        return _status_badge(classify_health(value, healthy=1.5, warning=1.0))
```

---

#### Change 6: Refactor `p2_debt_equity_status`

This one uses `higher_is_better=False` because lower Debt/Equity is healthier.

**BEFORE:**
```python
    @render.ui
    def p2_debt_equity_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Debt/Equity Ratio"].iloc[0]
        if value <= 1.0:
            return ui.tags.span("\u2713", class_="kpi-status healthy")
        elif value <= 2.0:
            return ui.tags.span("!", class_="kpi-status warning")
        else:
            return ui.tags.span("\u2717", class_="kpi-status danger")
```

**AFTER:**
```python
    @render.ui
    def p2_debt_equity_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Debt/Equity Ratio"].iloc[0]
        return _status_badge(
            classify_health(value, healthy=1.0, warning=2.0, higher_is_better=False)
        )
```

---

#### Change 7: Refactor `p2_cash_flows` to use `format_currency`

**BEFORE:**
```python
    @render.ui
    def p2_cash_flows():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.div("N/A")
        row = filtered.iloc[0]
        op = row["Cash Flow from Operating"]
        inv = row["Cash Flow from Investing"]
        fin = row["Cash Flow from Financial Activities"]
        def fmt(v):
            return f"-${abs(v):,.0f}M" if v < 0 else f"${v:,.0f}M"
        return ui.div(
            ui.span(f"Operating: {fmt(op)}", class_="kpi-label"),
            ui.br(),
            ui.span(f"Investing: {fmt(inv)}", class_="kpi-label"),
            ui.br(),
            ui.span(f"Financing: {fmt(fin)}", class_="kpi-label"),
            style="text-align: center;",
        )
```

**AFTER:**
```python
    @render.ui
    def p2_cash_flows():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.div("N/A")
        row = filtered.iloc[0]
        op = row["Cash Flow from Operating"]
        inv = row["Cash Flow from Investing"]
        fin = row["Cash Flow from Financial Activities"]
        return ui.div(
            ui.span(f"Operating: {format_currency(op)}", class_="kpi-label"),
            ui.br(),
            ui.span(f"Investing: {format_currency(inv)}", class_="kpi-label"),
            ui.br(),
            ui.span(f"Financing: {format_currency(fin)}", class_="kpi-label"),
            style="text-align: center;",
        )
```

---

### Complete final `src/pages/company.py` after all changes

For reference, here is the **complete file** after all refactoring. You can use this to verify your changes are correct, or copy it directly if you prefer.

In [ ]:
"""Page 2: Company Financial Health — UI layout and server logic."""

from shiny import reactive, render, ui
from shinywidgets import output_widget, render_altair

from charts.altair_charts import (
    build_cash_flows,
    build_ratio_over_time,
    build_revenue_over_time,
)
from components.empty_chart import empty_chart
from components.health_status import classify_health, format_currency
from data import ALL_SECTORS, CATEGORY_COMPANIES, YEAR_MIN, YEAR_MAX, tbl


def company_ui():
    """Return the full Page 2 layout."""
    # Sidebar inputs
    category_select = ui.input_select(
        id="category",
        label="Industry",
        choices=ALL_SECTORS,
        selected=ALL_SECTORS[0],
    )
    company_select = ui.input_select(
        id="company",
        label="Company",
        choices=[],
    )
    year_select = ui.output_ui("p2_year_slider")
    sidebar = ui.sidebar(
        ui.h4("Analytics Filters"),
        category_select,
        company_select,
        year_select,
        open="desktop",
    )

    # Profitability cards — mirror Financial Health layout (KPI + status + chart)
    card_npm = ui.card(
        ui.card_header("Net Profit Margin"),
        ui.div(
            ui.tags.h3(
                ui.output_text("p2_npm", inline=True),
                class_="kpi-value",
                style="display: inline;",
            ),
            ui.output_ui("p2_npm_status", style="display: inline;"),
            class_="kpi-value-row",
        ),
        output_widget("p2_npm_chart"),
        full_screen=True,
    )
    card_roe = ui.card(
        ui.card_header("Return on Equity (ROE)"),
        ui.div(
            ui.tags.h3(
                ui.output_text("p2_roe", inline=True),
                class_="kpi-value",
                style="display: inline;",
            ),
            ui.output_ui("p2_roe_status", style="display: inline;"),
            class_="kpi-value-row",
        ),
        output_widget("p2_roe_chart"),
        full_screen=True,
    )
    card_rev_income = ui.card(
        ui.card_header("Revenue & Net Income"),
        ui.output_ui("p2_rev_income_summary"),
        output_widget("p2_revenue_chart"),
        full_screen=True,
    )
    profitability_section = ui.div(
        ui.div("PROFITABILITY", class_="section-label section-label-blue"),
        ui.layout_columns(
            card_npm,
            card_roe,
            card_rev_income,
            col_widths=[4, 4, 4],
        ),
        class_="grid-section grid-section-blue",
    )

    # Financial Health cards (reactive outputs)
    card_current_ratio = ui.card(
        ui.card_header("Current Ratio"),
        ui.div(
            ui.tags.h3(
                ui.output_text("p2_current_ratio", inline=True),
                class_="kpi-value",
                style="display: inline;",
            ),
            ui.output_ui("p2_current_ratio_status", style="display: inline;"),
            class_="kpi-value-row",
        ),
        output_widget("p2_current_ratio_chart"),
        full_screen=True,
    )
    card_debt_equity = ui.card(
        ui.card_header("Debt / Equity Ratio"),
        ui.div(
            ui.tags.h3(
                ui.output_text("p2_debt_equity", inline=True),
                class_="kpi-value",
                style="display: inline;",
            ),
            ui.output_ui("p2_debt_equity_status", style="display: inline;"),
            class_="kpi-value-row",
        ),
        output_widget("p2_debt_equity_chart"),
        full_screen=True,
    )
    card_cash_flows = ui.card(
        ui.card_header("Cash Flows"),
        ui.output_ui("p2_cash_flows"),
        output_widget("p2_cash_flow_chart"),
        full_screen=True,
    )
    health_section = ui.div(
        ui.div("FINANCIAL HEALTH", class_="section-label section-label-red"),
        ui.layout_columns(
            card_current_ratio,
            card_debt_equity,
            card_cash_flows,
            col_widths=[4, 4, 4],
        ),
        class_="grid-section grid-section-red",
    )

    return ui.layout_sidebar(
        sidebar,
        ui.page_fillable(
            ui.h2("Financial Health Dashboard"),
            profitability_section,
            health_section,
        ),
    )


def company_server(input, output, session):
    """Page 2 reactive logic."""

    @reactive.effect
    @reactive.event(input.category)
    def _update_company_choices():
        companies = CATEGORY_COMPANIES.get(input.category(), [])
        selected = companies[0] if companies else None
        ui.update_select("company", choices=companies, selected=selected)

    @render.ui
    def p2_year_slider():
        company = input.company()
        # Use ibis to get year range for the selected company
        company_expr = tbl.filter(tbl["Company"] == company)
        company_data = company_expr.to_pandas()
        if company_data.empty:
            year_min = YEAR_MIN
            year_max = YEAR_MAX
        else:
            year_min = int(company_data["Year"].min())
            year_max = int(company_data["Year"].max())
        if year_min == year_max:
            return ui.div(
                ui.tags.label("Year", class_="control-label"),
                ui.tags.p(str(year_max), style="font-weight: 600; font-size: 1.1rem;"),
                ui.input_slider(
                    id="year", label="", min=year_min, max=year_max,
                    value=year_max, sep="",
                ),
                ui.tags.style("#year-label { display: none; } #year .irs { display: none; }"),
            )
        return ui.input_slider(
            id="year", label="Year", min=year_min, max=year_max,
            value=year_max, sep="",
        )

    @reactive.calc
    def p2_filtered_data():
        """Filter via ibis expressions, then materialize to pandas."""
        category = input.category()
        company = input.company()
        year = input.year()
        expr = tbl.filter(
            tbl["Category"] == category,
            tbl["Company"] == company,
            tbl["Year"] == year,
        )
        return expr.to_pandas()

    @reactive.calc
    def p2_company_data():
        """All rows for the selected company (for trend charts), via ibis."""
        company = input.company()
        return tbl.filter(tbl["Company"] == company).to_pandas()

    # --- Profitability KPIs ---

    @render.text
    def p2_npm():
        filtered = p2_filtered_data()
        if filtered.empty:
            return "N/A"
        value = filtered["Net Profit Margin"].iloc[0]
        return f"{value:.1f}%"

    @render.text
    def p2_roe():
        filtered = p2_filtered_data()
        if filtered.empty:
            return "N/A"
        value = filtered["ROE"].iloc[0]
        return f"{value:.2f}%"

    # Status icon mapping for health classification
    _STATUS_ICONS = {"healthy": "\u2713", "warning": "!", "danger": "\u2717"}

    def _status_badge(status: str):
        icon = _STATUS_ICONS[status]
        return ui.tags.span(icon, class_=f"kpi-status {status}")

    @render.ui
    def p2_npm_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Net Profit Margin"].iloc[0]
        return _status_badge(classify_health(value, healthy=10.0, warning=0.0))

    @render_altair
    def p2_npm_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_ratio_over_time(company_data, company, "Net Profit Margin")

    @render.ui
    def p2_roe_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["ROE"].iloc[0]
        return _status_badge(classify_health(value, healthy=15.0, warning=0.0))

    @render_altair
    def p2_roe_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_ratio_over_time(company_data, company, "ROE")

    @render.ui
    def p2_rev_income_summary():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.div("N/A")
        rev = filtered["Revenue"].iloc[0]
        ni = filtered["Net Income"].iloc[0]
        return ui.div(
            ui.span(f"Revenue: ${rev:,.0f}M", class_="kpi-label"),
            ui.span(" | ", style="color: var(--slate-400);"),
            ui.span(f"Net Income: ${ni:,.0f}M", class_="kpi-label"),
            style="padding: 0.25rem 0; text-align: center;",
        )

    @render_altair
    def p2_revenue_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_revenue_over_time(company_data, company)

    # --- Financial Health KPIs ---

    @render.text
    def p2_current_ratio():
        filtered = p2_filtered_data()
        if filtered.empty:
            return "N/A"
        value = filtered["Current Ratio"].iloc[0]
        return f"{value:.2f}"

    @render_altair
    def p2_current_ratio_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_ratio_over_time(company_data, company, "Current Ratio")

    @render.text
    def p2_debt_equity():
        filtered = p2_filtered_data()
        if filtered.empty:
            return "N/A"
        value = filtered["Debt/Equity Ratio"].iloc[0]
        return f"{value:.2f}"

    @render_altair
    def p2_debt_equity_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_ratio_over_time(company_data, company, "Debt/Equity Ratio")

    @render.ui
    def p2_cash_flows():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.div("N/A")
        row = filtered.iloc[0]
        op = row["Cash Flow from Operating"]
        inv = row["Cash Flow from Investing"]
        fin = row["Cash Flow from Financial Activities"]
        return ui.div(
            ui.span(f"Operating: {format_currency(op)}", class_="kpi-label"),
            ui.br(),
            ui.span(f"Investing: {format_currency(inv)}", class_="kpi-label"),
            ui.br(),
            ui.span(f"Financing: {format_currency(fin)}", class_="kpi-label"),
            style="text-align: center;",
        )

    @render.ui
    def p2_current_ratio_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Current Ratio"].iloc[0]
        return _status_badge(classify_health(value, healthy=1.5, warning=1.0))

    @render.ui
    def p2_debt_equity_status():
        filtered = p2_filtered_data()
        if filtered.empty:
            return ui.tags.span()
        value = filtered["Debt/Equity Ratio"].iloc[0]
        return _status_badge(
            classify_health(value, healthy=1.0, warning=2.0, higher_is_better=False)
        )

    @render_altair
    def p2_cash_flow_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_cash_flows(company_data, company)

---

### Step 4: Create `tests/test_health_status.py`

Create the test file with comprehensive tests for both `classify_health` and `format_currency`. The tests are organized into 3 classes covering:
- `classify_health` with `higher_is_better=True` (default) — NPM, ROE, Current Ratio thresholds
- `classify_health` with `higher_is_better=False` — Debt/Equity thresholds
- `format_currency` — positive, negative, zero, and large values

In [ ]:
"""Unit tests for health status classification and formatting helpers."""

import pytest
from components.health_status import classify_health, format_currency


# --- classify_health tests (higher_is_better=True) ---

class TestClassifyHealthHigherIsBetter:
    """Tests for classify_health with higher_is_better=True (default)."""

    def test_healthy_above_threshold(self):
        assert classify_health(15.0, healthy=10.0, warning=0.0) == "healthy"

    def test_healthy_at_threshold(self):
        assert classify_health(10.0, healthy=10.0, warning=0.0) == "healthy"

    def test_warning_between_thresholds(self):
        assert classify_health(5.0, healthy=10.0, warning=0.0) == "warning"

    def test_warning_at_threshold(self):
        assert classify_health(0.0, healthy=10.0, warning=0.0) == "warning"

    def test_danger_below_warning(self):
        assert classify_health(-5.0, healthy=10.0, warning=0.0) == "danger"

    def test_npm_healthy(self):
        """Net Profit Margin >= 10% is healthy."""
        assert classify_health(12.5, healthy=10.0, warning=0.0) == "healthy"

    def test_npm_warning(self):
        """Net Profit Margin 0-10% is warning."""
        assert classify_health(5.0, healthy=10.0, warning=0.0) == "warning"

    def test_npm_danger(self):
        """Net Profit Margin < 0% is danger."""
        assert classify_health(-3.0, healthy=10.0, warning=0.0) == "danger"

    def test_roe_healthy(self):
        """ROE >= 15% is healthy."""
        assert classify_health(20.0, healthy=15.0, warning=0.0) == "healthy"

    def test_current_ratio_healthy(self):
        """Current Ratio >= 1.5 is healthy."""
        assert classify_health(2.0, healthy=1.5, warning=1.0) == "healthy"

    def test_current_ratio_warning(self):
        """Current Ratio 1.0-1.5 is warning."""
        assert classify_health(1.2, healthy=1.5, warning=1.0) == "warning"

    def test_current_ratio_danger(self):
        """Current Ratio < 1.0 is danger."""
        assert classify_health(0.8, healthy=1.5, warning=1.0) == "danger"


# --- classify_health tests (higher_is_better=False) ---

class TestClassifyHealthLowerIsBetter:
    """Tests for classify_health with higher_is_better=False (e.g., Debt/Equity)."""

    def test_healthy_below_threshold(self):
        assert classify_health(0.5, healthy=1.0, warning=2.0, higher_is_better=False) == "healthy"

    def test_healthy_at_threshold(self):
        assert classify_health(1.0, healthy=1.0, warning=2.0, higher_is_better=False) == "healthy"

    def test_warning_between_thresholds(self):
        assert classify_health(1.5, healthy=1.0, warning=2.0, higher_is_better=False) == "warning"

    def test_warning_at_threshold(self):
        assert classify_health(2.0, healthy=1.0, warning=2.0, higher_is_better=False) == "warning"

    def test_danger_above_warning(self):
        assert classify_health(3.0, healthy=1.0, warning=2.0, higher_is_better=False) == "danger"

    def test_debt_equity_healthy(self):
        """Debt/Equity <= 1.0 is healthy."""
        assert classify_health(0.8, healthy=1.0, warning=2.0, higher_is_better=False) == "healthy"

    def test_debt_equity_danger(self):
        """Debt/Equity > 2.0 is danger."""
        assert classify_health(5.0, healthy=1.0, warning=2.0, higher_is_better=False) == "danger"


# --- format_currency tests ---

class TestFormatCurrency:
    """Tests for format_currency."""

    def test_positive_value(self):
        assert format_currency(1234.0) == "$1,234M"

    def test_negative_value(self):
        assert format_currency(-567.0) == "-$567M"

    def test_zero(self):
        assert format_currency(0.0) == "$0M"

    def test_large_value(self):
        assert format_currency(120000.0) == "$120,000M"

    def test_small_negative(self):
        assert format_currency(-0.5) == "-$0M"

**What these tests cover:**
- **`TestClassifyHealthHigherIsBetter`** (12 tests): boundary values (at threshold, above, below) for NPM, ROE, and Current Ratio metrics
- **`TestClassifyHealthLowerIsBetter`** (7 tests): boundary values for Debt/Equity where lower values are healthier
- **`TestFormatCurrency`** (5 tests): positive, negative, zero, large, and small-negative dollar formatting

> **Note:** The import path `from components.health_status import ...` works because `pyproject.toml` has `pythonpath = ["src"]` configured for pytest.

---

### Step 5: Test locally

Run linting and tests to make sure everything passes:

```bash
conda run -n fin-health ruff check src/ tests/ --fix && ruff format src/ tests/
conda run -n fin-health pytest tests/ -v
```

**Expected output:** All tests should pass, including the new `tests/test_health_status.py` tests (24 total new tests) and the existing `tests/test_data.py` tests.

Also verify the app still works:

```bash
conda run -n fin-health shiny run src/app.py
```

**Verify:**
- [ ] App starts without errors
- [ ] Company page health status icons still display correctly (checkmark, exclamation, X)
- [ ] Cash flow values still format correctly (e.g., `$1,234M`, `-$567M`)
- [ ] All status colors (green/yellow/red) match the previous behavior

---

### Step 6: Commit and create PR

```bash
git add src/components/health_status.py src/pages/company.py tests/test_health_status.py
git commit -m "feat: refactor health classification into testable pure functions with unit tests"
git push origin feat/unit-tests
```

Create PR: Base `main` <- Compare `feat/unit-tests`  
Title: `feat: Refactor health status classification + unit tests`

Description:
```markdown
Fixes #<task-5-issue-number>

### Proposed Changes
- Created `src/components/health_status.py` with two pure functions:
  - `classify_health(value, healthy, warning, higher_is_better)` — classifies a metric into healthy/warning/danger
  - `format_currency(value)` — formats dollar values as "$1,234M" or "-$567M"
- Refactored 4 status functions in `company.py` to use `classify_health()` instead of inline if/elif/else
- Refactored `p2_cash_flows` to use `format_currency()` instead of inline `fmt()` closure
- Added `tests/test_health_status.py` with 24 unit tests covering boundary values and real metric thresholds
- All tests pass on clean environment
```

Get team review, then merge.

```bash
git checkout main
git pull origin main
git branch -D feat/unit-tests
```

---

## Part 2: CHANGELOG (Task 7 — Luke's portion)

Write the **Added**, **Changed**, **Fixed**, and **Known Issues** sections for the v0.4.0 CHANGELOG entry. Eden will add the remaining sections (Release Highlight, Collaboration, Reflection) separately.

### Step 1: Create the branch

```bash
git checkout main
git pull origin main
git checkout -b docs/changelog-v0.4.0
git push origin docs/changelog-v0.4.0
```

> **Coordinate with Eden:** If Eden has already created this branch, pull it instead of creating a new one:
> ```bash
> git fetch origin
> git checkout docs/changelog-v0.4.0
> git pull origin docs/changelog-v0.4.0
> ```

---

### Step 2: Add the v0.4.0 sections to `CHANGELOG.md`

Open `CHANGELOG.md` and add the following block **above** the existing `## [v0.3.0]` entry. Leave placeholder comments where Eden will add their sections.

In [ ]:
## [v0.4.0] - (2026-03-18)

### Added
- **Parquet + DuckDB Backend**: Migrated data loading to `ibis.duckdb.connect()` + `con.read_parquet()` for lazy query execution and improved performance.
- **RAG Finance Glossary**: Integrated a knowledge base (`knowledge/glossary.md`) into querychat for retrieval-augmented financial term definitions.
- **Playwright Behavior Tests**: Added 3 browser-based behavior tests covering distinct dashboard interactions.
- **Unit Tests**: Added pytest unit tests for refactored `classify_health()` and `format_currency()` helper functions.

### Changed
- **Data Pipeline**: Replaced CSV-based `pandas.read_csv()` with parquet-based `ibis` expressions; all filtering now happens at the database level before materializing to pandas.
- **Health Status Refactor**: Extracted repeated threshold-based health classification logic from `company.py` into a reusable, testable `classify_health()` pure function in `components/health_status.py`.
- **Currency Formatting**: Extracted inline `fmt()` closure into a standalone `format_currency()` helper for consistency and testability.
- **Dependencies**: Added `ibis-framework[duckdb]` and `playwright` to `requirements.txt` and `environment.yml`.

### Fixed
- (Items from feedback prioritization will go here — TBD based on M4 Feedback Issue)

### Known Issues
- **fin-chat requires API token**: The fin-chat page requires a `GITHUB_TOKEN` environment variable; without it, a fallback message is displayed.
- **Querychat latency**: LLM-powered queries may take 2-5 seconds depending on API response time.

**What each section covers:**

- **Added** — New features introduced in M4: parquet/DuckDB backend, RAG glossary, Playwright behavior tests, unit tests
- **Changed** — Modifications to existing functionality: data pipeline migration, health status refactor, currency formatting extraction, dependency updates
- **Fixed** — Placeholder for bug fixes from the M4 feedback prioritization issue (fill in once feedback items are resolved)
- **Known Issues** — Documented limitations: API token requirement for fin-chat, querychat response latency

> **Note:** Eden will add the **Release Highlight**, **Collaboration**, and **Reflection** sections. Do not add those — just the 4 sections above.

---

### Step 3: Commit and create PR

```bash
git add CHANGELOG.md
git commit -m "docs: add Added/Changed/Fixed/Known Issues for v0.4.0 CHANGELOG"
git push origin docs/changelog-v0.4.0
```

Create PR: Base `main` <- Compare `docs/changelog-v0.4.0`  
Title: `docs: CHANGELOG v0.4.0 — Added/Changed/Fixed/Known Issues`

Description:
```markdown
Fixes #<task-7-issue-number>

### Proposed Changes
- Added v0.4.0 CHANGELOG entry with Added, Changed, Fixed, and Known Issues sections
- Eden will add Release Highlight, Collaboration, and Reflection sections in a follow-up commit
```

> **Do not merge yet** — wait for Eden to add their sections to the same branch, then merge together.

```bash
git checkout main
git pull origin main
git branch -D docs/changelog-v0.4.0
```

---

## Review Teammate PRs

You'll be reviewing PRs from other team members when requested.

**When reviewing:**
- Pull the branch locally and run `shiny run src/app.py`
- Verify new functionality works
- Run: `ruff check src/` and `pytest tests/ -v`
- Check imports reference the module structure (`from data import ...`, `from components.health_status import ...`)
- Provide constructive feedback